# Google Colab notebook для GPU speaker diarization (Питер FM)

Этот ноутбук запускает **только diarization-этап** поверх уже готовых ASR-артефактов
тестового фильма «Питер FM» ().

Что делает ноутбук:
- читает уже существующие  из ASR run;
- сохраняет  по каждому чанку;
- пишет .

Предполагается, что ASR-ноутбук уже отработал и чанки лежат в:



## 1. Проверка runtime

Перед запуском выберите в Colab:

**Runtime → Change runtime type → T4 GPU**


In [1]:
!nvidia-smi

import sys
import torch

print("python:", sys.version)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


Sun Mar 29 11:42:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Подключение Google Drive

Рекомендуется хранить на Google Drive:
- код репозитория отдельно;
- ASR-артефакты отдельно;
- выход diarization рядом с ASR-артефактами или в отдельной папке.


In [2]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


## 3. Получение кода из GitHub

Если репозиторий уже клонирован, ноутбук просто подтянет последние изменения.


In [3]:
from pathlib import Path

REPO_URL = "https://github.com/DeMiYak/AudioIntent.git"
REPO_DIR = Path("/content/AudioIntent")

if REPO_DIR.exists():
    print("Repository already exists. Pulling latest changes...")
    !git -C "{REPO_DIR}" pull --ff-only
else:
    !git clone "{REPO_URL}" "{REPO_DIR}"

!git -C "{REPO_DIR}" rev-parse --abbrev-ref HEAD
!git -C "{REPO_DIR}" log -1 --oneline


Cloning into '/content/AudioIntent'...
remote: Enumerating objects: 184, done.
remote: Counting objects: 100% (184/184), done.
remote: Compressing objects: 100% (127/127), done.
remote: Total 184 (delta 98), reused 129 (delta 46), pack-reused 0 (from 0)
Receiving objects: 100% (184/184), 162.69 KiB | 1.52 MiB/s, done.
Resolving deltas: 100% (98/98), done.
main
ae68f93 (HEAD -> main, origin/main, origin/HEAD) Adapt for community-1


## 4. Пути к ASR-артефактам

Ноутбук ожидает готовую структуру от ASR-ноутбука:

- 
- 

 по умолчанию пишется **в те же папки чанков** ().


In [4]:
from pathlib import Path
import os

DRIVE_BASE = Path("/content/drive/MyDrive")
PROJECT_ROOT_ON_DRIVE = DRIVE_BASE / "AudioIntent"

ASR_OUTPUT_DIR = PROJECT_ROOT_ON_DRIVE / "data" / "artifacts" / "test_piter_fm_asr_diarization_colab"
WINDOWS_DIR = ASR_OUTPUT_DIR / "windows"

# If True, diarization.json is written directly into windows/chunk_NNN/
WRITE_IN_PLACE = True

DIARIZATION_OUTPUT_ROOT = PROJECT_ROOT_ON_DRIVE / "data" / "artifacts" / "test_piter_fm_diarization_colab"

VENV_DIR = Path("/content/venvs/pyannote_diar")
VENV_PYTHON = VENV_DIR / "bin" / "python"

print("WINDOWS_DIR exists:", WINDOWS_DIR.exists())
if WINDOWS_DIR.exists():
    chunks = sorted([p for p in WINDOWS_DIR.iterdir() if p.is_dir()])
    print(f"Chunks found: {len(chunks)}")
    for c in chunks[:3]:
        print(f"  {c.name}: audio.wav={( c / 'audio.wav').exists()}, transcript.json={(c / 'transcript.json').exists()}")


PROJECT_ROOT_ON_DRIVE: /content/drive/MyDrive/AudioIntent
ASR_OUTPUT_DIR: /content/drive/MyDrive/AudioIntent/data/artifacts/validation_status_svoboden_asr_colab
WINDOWS_DIR exists: True
VENV_DIR: /content/venvs/pyannote_diar


## 5. Создание отдельного `venv` для diarization

Этот notebook **не ставит `pyannote.audio` в системный Colab runtime**.  
Вместо этого создаётся отдельный `venv`, и все тяжёлые зависимости ставятся туда.

Плюсы:
- `pip` не пытается согласовать весь предустановленный Colab-стек;
- не нужны постоянные `Restart session`;
- `pyannote` и его зависимости изолированы от notebook kernel.

По умолчанию `venv` создаётся в `/content/venvs/...` — это **временная файловая система runtime**.  
Если хотите хранить `venv` в Google Drive, перенесите `VENV_DIR` в `/content/drive/MyDrive/...`, но это обычно медленнее.

In [6]:
%%bash
set -euo pipefail

# Install system dependencies
apt-get -qq update
apt-get -qq install -y ffmpeg python3-venv wget

VENV_DIR="/content/venvs/pyannote_diar"

# Create venv without pip to avoid the 'ensurepip' error in Colab
python3 -m venv --without-pip "$VENV_DIR"

# Manually install pip into the venv
wget -q https://bootstrap.pypa.io/get-pip.py
"$VENV_DIR/bin/python" get-pip.py
rm get-pip.py

# Install requirements
# We pin setuptools<82 to avoid conflicts with torch 2.11.0
"$VENV_DIR/bin/python" -m pip install -U pip "setuptools<82" wheel
"$VENV_DIR/bin/python" -m pip install --no-cache-dir "pyannote.audio==4.0.4" soundfile

  Using cached pip-26.0.1-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-26.0.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 26.0.1
    Uninstalling pip-26.0.1:
      Successfully uninstalled pip-26.0.1
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 893.7/893.7 kB 234.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 474.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 447.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 222.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 529.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


## 6. Проверка импорта внутри `venv`

In [7]:
%%bash
set -euo pipefail

VENV_DIR="/content/venvs/pyannote_diar"

# Force a non-interactive backend for matplotlib inside the venv
export MPLBACKEND=Agg

"$VENV_DIR/bin/python" - <<'PY'
import sys
import torch
import pyannote.audio

print("python:", sys.version)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("pyannote.audio:", pyannote.audio.__version__)
PY

python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
torch: 2.11.0+cu130
cuda available: True
device: Tesla T4
pyannote.audio: 4.0.4


/content/venvs/pyannote_diar/lib/python3.12/site-packages/pyannote/audio/core/io.py:47: UserWarning: 
torchcodec is not installed correctly so built-in audio decoding will fail. Solutions are:
* use audio preloaded in-memory as a {'waveform': (channel, time) torch.Tensor, 'sample_rate': int} dictionary;
* fix torchcodec installation. Error message was:

Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.11.0+cu130) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
           

## 7. Загрузка Hugging Face token из Colab Secrets

In [8]:
from google.colab import userdata
import os

SECRET_NAME = "AudioIntent"

try:
    os.environ["HF_TOKEN"] = userdata.get(SECRET_NAME)
    print(f"HF_TOKEN loaded from Colab Secrets: {SECRET_NAME}")
except userdata.SecretNotFoundError:
    print(f"Secret '{SECRET_NAME}' not found in Colab Secrets.")
    print("Создайте secret с именем 'AudioIntent' или задайте os.environ['HF_TOKEN'] вручную.")
except Exception as e:
    print("Не удалось загрузить HF_TOKEN из Colab Secrets:", repr(e))
    print("Задайте os.environ['HF_TOKEN'] вручную, если нужно.")

HF_TOKEN loaded from Colab Secrets: AudioIntent


## 8. Генерация standalone runner-скрипта для diarization

In [9]:
from pathlib import Path

RUNNER_SCRIPT = REPO_DIR / "src" / "scripts" / "run_diarization_batch_colab.py"
print("Runner script exists:", RUNNER_SCRIPT.exists())
if not RUNNER_SCRIPT.exists():
    raise FileNotFoundError(
        f"Runner script not found: {RUNNER_SCRIPT}
"
        "Make sure the repo was cloned correctly."
    )

Runner script written to: /content/AudioIntent/scripts/run_diarization_batch_colab.py


## 9. Smoke test на одном окне

In [15]:
import os
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/AudioIntent")
VENV_PYTHON = Path("/content/venvs/pyannote_diar/bin/python")
RUNNER_PATH = REPO_DIR / "src" / "scripts" / "run_diarization_batch_colab.py"

# Uses WINDOWS_DIR and DIARIZATION_OUTPUT_ROOT from Cell 8
SELECTED_WINDOW_ID = "chunk_001"

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN не найден в os.environ")

window_dir = WINDOWS_DIR / SELECTED_WINDOW_ID
audio_path = window_dir / "audio.wav"

print("window_dir exists:", window_dir.exists())
print("audio.wav exists:", audio_path.exists())
print("runner exists:", RUNNER_PATH.exists())
print("venv python exists:", VENV_PYTHON.exists())

command = [
    str(VENV_PYTHON),
    str(RUNNER_PATH),
    "--windows-dir", str(WINDOWS_DIR),
    "--output-root", str(DIARIZATION_OUTPUT_ROOT),
    "--write-in-place",
    "--model-name", "pyannote/speaker-diarization-community-1",
    "--device", "cuda",
    "--min-speakers", "2",
    "--max-speakers", "6",
    "--single-window-id", SELECTED_WINDOW_ID,
    "--use-exclusive",
    "--force-rerun"
]

env = os.environ.copy()
env["HF_TOKEN"] = hf_token
env["PYTHONPATH"] = f"{REPO_DIR}:{env.get('PYTHONPATH', '')}"
env["MPLBACKEND"] = "Agg"

print("
Running command:")
print(" ".join(command))

result = subprocess.run(
    command,
    env=env,
    cwd=str(REPO_DIR),
    text=True,
    capture_output=True,
)

print("
Return code:", result.returncode)
print("
--- STDOUT ---
")
print(result.stdout if result.stdout else "<empty>")
print("
--- STDERR ---
")
print(result.stderr if result.stderr else "<empty>")

if result.returncode != 0:
    raise RuntimeError("Smoke test failed. See STDERR above.")


window_dir exists: True
audio.wav exists: True
runner exists: True
venv python exists: True

Running command:
/content/venvs/pyannote_diar/bin/python /content/AudioIntent/scripts/run_diarization_batch_colab.py --windows-dir /content/drive/MyDrive/AudioIntent/data/artifacts/validation_status_svoboden_asr_colab/windows --output-root /content/drive/MyDrive/AudioIntent/data/artifacts/validation_status_svoboden_diarization_colab --write-in-place --model-name pyannote/speaker-diarization-community-1 --device cuda --min-speakers 2 --max-speakers 4 --single-window-id val_001_21001 --use-exclusive --force-rerun

Return code: 0

--- STDOUT ---

[INFO] [1/1] window_id=val_001_21001
[INFO] Loading pyannote pipeline with token=... | model=pyannote/speaker-diarization-community-1
[INFO] Pipeline moved to CUDA.
[INFO] Running diarization for: audio.wav
[INFO] inference kwargs: {'min_speakers': 2, 'max_speakers': 4}
[FALLBACK] Using in-memory audio input to bypass pyannote/torchcodec file decoding.
se

## 10. Batch diarization по всем validation-окнам

In [16]:
import os
import subprocess

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN не найден. Сначала задайте Hugging Face token.")

MODEL_NAME = "pyannote/speaker-diarization-community-1"
DEVICE = "cuda"
USE_EXCLUSIVE = True

# Рекомендуемый режим для batch:
# не фиксировать точное число спикеров, а задать нижнюю/верхнюю границы
NUM_SPEAKERS = None
MIN_SPEAKERS = 2
MAX_SPEAKERS = 6

FORCE_RERUN = False
LIMIT = None   # например 10 для тестового batch; None = все окна

if NUM_SPEAKERS is not None and (MIN_SPEAKERS is not None or MAX_SPEAKERS is not None):
    raise ValueError("Используйте либо NUM_SPEAKERS, либо MIN/MAX_SPEAKERS, но не оба режима сразу.")

print("Batch diarization config:")
print("  windows_dir:", WINDOWS_DIR)
print("  output_root:", DIARIZATION_OUTPUT_ROOT)
print("  model_name:", MODEL_NAME)
print("  device:", DEVICE)
print("  use_exclusive:", USE_EXCLUSIVE)
print("  write_in_place:", WRITE_IN_PLACE)
print("  num_speakers:", NUM_SPEAKERS)
print("  min_speakers:", MIN_SPEAKERS)
print("  max_speakers:", MAX_SPEAKERS)
print("  force_rerun:", FORCE_RERUN)
print("  limit:", LIMIT)

command = [
    str(VENV_PYTHON),
    str(RUNNER_SCRIPT),
    "--windows-dir", str(WINDOWS_DIR),
    "--output-root", str(DIARIZATION_OUTPUT_ROOT),
    "--model-name", MODEL_NAME,
    "--device", DEVICE,
]

if WRITE_IN_PLACE:
    command.append("--write-in-place")
if USE_EXCLUSIVE:
    command.append("--use-exclusive")
if FORCE_RERUN:
    command.append("--force-rerun")

if NUM_SPEAKERS is not None:
    command += ["--num-speakers", str(NUM_SPEAKERS)]
else:
    if MIN_SPEAKERS is not None:
        command += ["--min-speakers", str(MIN_SPEAKERS)]
    if MAX_SPEAKERS is not None:
        command += ["--max-speakers", str(MAX_SPEAKERS)]

if LIMIT is not None:
    command += ["--limit", str(LIMIT)]

env = os.environ.copy()
env["HF_TOKEN"] = HF_TOKEN
env["PYTHONPATH"] = f"{REPO_DIR}:{env.get('PYTHONPATH', '')}"
env["MPLBACKEND"] = "Agg"

print("\nRunning batch command:")
print(" ".join(command))

subprocess.run(
    command,
    check=True,
    env=env,
    cwd=str(REPO_DIR),
)

Batch diarization config:
  windows_dir: /content/drive/MyDrive/AudioIntent/data/artifacts/validation_status_svoboden_asr_colab/windows
  output_root: /content/drive/MyDrive/AudioIntent/data/artifacts/validation_status_svoboden_diarization_colab
  model_name: pyannote/speaker-diarization-community-1
  device: cuda
  use_exclusive: True
  write_in_place: True
  num_speakers: None
  min_speakers: 2
  max_speakers: 4
  force_rerun: False
  limit: None

Running batch command:
/content/venvs/pyannote_diar/bin/python /content/AudioIntent/scripts/run_diarization_batch_colab.py --windows-dir /content/drive/MyDrive/AudioIntent/data/artifacts/validation_status_svoboden_asr_colab/windows --output-root /content/drive/MyDrive/AudioIntent/data/artifacts/validation_status_svoboden_diarization_colab --model-name pyannote/speaker-diarization-community-1 --device cuda --write-in-place --use-exclusive --min-speakers 2 --max-speakers 4


CompletedProcess(args=['/content/venvs/pyannote_diar/bin/python', '/content/AudioIntent/scripts/run_diarization_batch_colab.py', '--windows-dir', '/content/drive/MyDrive/AudioIntent/data/artifacts/validation_status_svoboden_asr_colab/windows', '--output-root', '/content/drive/MyDrive/AudioIntent/data/artifacts/validation_status_svoboden_diarization_colab', '--model-name', 'pyannote/speaker-diarization-community-1', '--device', 'cuda', '--write-in-place', '--use-exclusive', '--min-speakers', '2', '--max-speakers', '4'], returncode=0)

## 11. Быстрая проверка результатов

In [17]:
import json

windows = sorted([p for p in WINDOWS_DIR.iterdir() if p.is_dir()])
payloads = []

for window_dir in windows:
    diar_path = window_dir / "diarization.json"
    if diar_path.exists():
        with diar_path.open("r", encoding="utf-8") as f:
            payloads.append((window_dir.name, json.load(f)))

print("diarization.json count:", len(payloads))

if payloads:
    fallback_auth = sum(1 for _, p in payloads if p.get("load_auth_fallback_used"))
    fallback_exclusive = sum(1 for _, p in payloads if p.get("exclusive_fallback_used"))
    print("auth fallback used:", fallback_auth)
    print("exclusive fallback used:", fallback_exclusive)
    print("sample window_id:", payloads[0][0])
    print("sample keys:", sorted(payloads[0][1].keys()))
    print("sample first segment:", payloads[0][1].get("segments", [])[:1])

summary_path = (ASR_OUTPUT_DIR / "diarization_run_summary.json") if WRITE_IN_PLACE else (DIARIZATION_OUTPUT_ROOT / "diarization_run_summary.json")
print("summary path:", summary_path)
print("summary exists:", summary_path.exists())
if summary_path.exists():
    with summary_path.open("r", encoding="utf-8") as f:
        summary = json.load(f)
    print("num_ok:", summary.get("num_ok"))
    print("num_errors:", summary.get("num_errors"))

diarization.json count: 28
auth fallback used: 0
exclusive fallback used: 0
sample window_id: val_001_21001
sample keys: ['audio_path', 'device', 'exclusive_fallback_used', 'exclusive_segments', 'load_auth_fallback_used', 'load_auth_mode', 'model_name', 'num_speakers_detected', 'num_speakers_detected_exclusive', 'num_speakers_detected_regular', 'regular_segments', 'segments', 'selected_segments_mode', 'used_exclusive_diarization']
sample first segment: [{'segment_id': 'spkseg_0001', 'start_time': 0.031, 'end_time': 1.027, 'speaker_label': 'SPEAKER_00'}]
summary path: /content/drive/MyDrive/AudioIntent/data/artifacts/validation_status_svoboden_asr_colab/diarization_run_summary.json
summary exists: True
num_ok: 1
num_errors: 0


## 12. Архивация результатов



In [18]:
from pathlib import Path

TO_ZIP = ASR_OUTPUT_DIR if WRITE_IN_PLACE else DIARIZATION_OUTPUT_ROOT
ZIP_PATH = Path(str(TO_ZIP) + ".zip")

%cd {PROJECT_ROOT_ON_DRIVE}
!zip -r "{ZIP_PATH}" "{TO_ZIP}"

print("Zip created:", ZIP_PATH)

/content/drive/MyDrive/AudioIntent
updating: content/drive/MyDrive/AudioIntent/data/artifacts/validation_status_svoboden_asr_colab/ (stored 0%)
updating: content/drive/MyDrive/AudioIntent/data/artifacts/validation_status_svoboden_asr_colab/gold.xlsx (deflated 8%)
updating: content/drive/MyDrive/AudioIntent/data/artifacts/validation_status_svoboden_asr_colab/windows/ (stored 0%)
updating: content/drive/MyDrive/AudioIntent/data/artifacts/validation_status_svoboden_asr_colab/windows/val_001_21001/ (stored 0%)
updating: content/drive/MyDrive/AudioIntent/data/artifacts/validation_status_svoboden_asr_colab/windows/val_001_21001/audio.wav (deflated 20%)
updating: content/drive/MyDrive/AudioIntent/data/artifacts/validation_status_svoboden_asr_colab/windows/val_001_21001/transcript.json (deflated 72%)
updating: content/drive/MyDrive/AudioIntent/data/artifacts/validation_status_svoboden_asr_colab/windows/val_001_21001/summary.json (deflated 42%)
updating: content/drive/MyDrive/AudioIntent/data/a